# Семинар 7. От ближайших соседей к полноценному ML-исследованию

Сегодня построим классификатор видов пингвинов по числовым и категориальным признакам. Начнём с устройства kNN, затем последовательно пройдём исследование реальной таблицы: **EDA → baseline → подготовка признаков → обучение → настройка → диагностика → итоговый test → сохранение модели**.

По дороге изучим обучаемые преобразователи, `Pipeline` и `ColumnTransformer`. Каждая конструкция появится сначала как решение понятной практической задачи.

**Основная метрика сравнения — macro-F1.** Нас интересуют итоговые метки видов, причём каждый вид должен учитываться при оценке качества, даже если встречается реже других. Дополнительно будем смотреть accuracy, precision и recall по классам. Многоклассовые ROC-AUC и average precision оценивают качество ранжирования по классовым scores; это полезная дополнительная диагностика, но для сравнения итоговых классификационных решений здесь удобнее macro-F1.

## Часть I. Как kNN устроен внутри

Начнём с известного небольшого набора **Wine**: для образцов вина известны лабораторные измерения и один из трёх классов. Используем два непрерывных признака для рисунка, чтобы устройство алгоритма было видно на плоскости. Полноценная программа для работы с реальными таблицами появится дальше.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

SEED = 42
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.5)

### Небольшая выборка для проверки алгоритма

Используем содержание алкоголя (`alcohol`) и количество флавоноидов (`flavanoids`). Значения двух признаков имеют сопоставимый порядок величины, поэтому их можно показать на одном рисунке с **одинаковым масштабом осей**. На плоскости обычная евклидова окружность должна выглядеть окружностью: иначе ближайшие точки кажутся расположенными нелогично.

`load_wine` возвращает числовую таблицу без пропусков. Здесь нам важен сам механизм поиска соседей.

In [ ]:
wine = load_wine(as_frame=True)
wine_features = ["alcohol", "flavanoids"]
X_wine = wine.frame[wine_features]
y_wine = wine.target

wine.frame[wine_features + ["target"]].head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for class_id, name in enumerate(wine.target_names):
    part = X_wine[y_wine == class_id]
    ax.scatter(part.iloc[:, 0], part.iloc[:, 1], alpha=0.7, label=name, s=30)

# Одна единица по обеим осям имеет одинаковую длину на экране.
ax.set_aspect("equal", adjustable="box")
ax.set(xlabel="Содержание алкоголя", ylabel="Количество флавоноидов",
       title="Wine: три класса в пространстве двух признаков")
ax.legend(title="Класс", loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

### Простейший классификатор: `fit`, `predict_proba`, `predict`

Рассмотрим алгоритм без встроенного масштабирования. Метод `fit` запоминает обучающие объекты и их метки. `predict_proba` находит ближайших соседей нового объекта и считает долю голосов каждого класса. `predict` выбирает класс с максимальной долей.

Код уже написан. Важно проследить **путь одного объекта через все методы**, а не запоминать синтаксис класса.

In [ ]:
class SimpleKNN:
    """Учебный kNN-классификатор с евклидовым расстоянием и равными голосами."""

    def __init__(self, k=5):
        self.k = k

    def fit(self, X_train, y_train):
        # Сохраняем обучающие данные. Никаких глобальных коэффициентов не ищем.
        self.X_train_ = np.asarray(X_train, dtype=float)
        self.y_train_ = np.asarray(y_train)
        self.classes_ = np.unique(self.y_train_)
        if not 1 <= self.k <= len(self.X_train_):
            raise ValueError("k должно лежать между 1 и числом обучающих объектов")
        return self

    def predict_proba(self, X_test):
        X_test = np.asarray(X_test, dtype=float)
        probabilities = np.zeros((len(X_test), len(self.classes_)))

        for row_idx, point in enumerate(X_test):
            # Вычисляем расстояния от нового объекта до каждой обучающей строки.
            distances = np.linalg.norm(self.X_train_ - point, axis=1)
            neighbor_idx = np.argsort(distances)[:self.k]
            neighbor_labels = self.y_train_[neighbor_idx]

            # Каждая доля = число голосов класса / k.
            for class_idx, class_label in enumerate(self.classes_):
                probabilities[row_idx, class_idx] = np.mean(
                    neighbor_labels == class_label
                )
        return probabilities

    def predict(self, X_test):
        probabilities = self.predict_proba(X_test)
        return self.classes_[np.argmax(probabilities, axis=1)]

### Разбираем один прогноз

Выделим обучающую и проверочную части. Выберем первый объект из validation и **вручную** вычислим расстояния до обучающих объектов. Затем отсортируем расстояния и получим индексы пяти ближайших соседей.

Важно: расстояния вычисляются в **числовых единицах признаков**, а не в пикселях на экране. Чтобы зрительное впечатление соответствовало вычислениям, для рисунка используем одинаковый масштаб по обеим осям.

In [ ]:
X_wine_train, X_wine_valid, y_wine_train, y_wine_valid = train_test_split(
    X_wine, y_wine, test_size=0.25, stratify=y_wine, random_state=SEED
)
query = X_wine_valid.iloc[0].to_numpy()
train_array = X_wine_train.to_numpy()

# Для каждого обучающего образца вычисляем длину разности двух векторов.
distances = np.linalg.norm(train_array - query, axis=1)
neighbor_positions = np.argsort(distances)[:5]

# Сортировка идёт по численным расстояниям, независимо от вида графика.
nearest_table = pd.DataFrame({
    "алкоголь": train_array[neighbor_positions, 0],
    "флавоноиды": train_array[neighbor_positions, 1],
    "расстояние": distances[neighbor_positions],
    "класс": [wine.target_names[i] for i in y_wine_train.iloc[neighbor_positions]],
})
nearest_table.round(3)

In [ ]:
from matplotlib.patches import Circle

fig, ax = plt.subplots(figsize=(8, 6))
for class_id, name in enumerate(wine.target_names):
    part = X_wine_train[y_wine_train == class_id]
    ax.scatter(part.iloc[:, 0], part.iloc[:, 1], alpha=0.42, s=30, label=name)

# Соединим новый объект с выбранными соседями: так видно, какое расстояние сравниваем.
for position in neighbor_positions:
    ax.plot([query[0], train_array[position, 0]],
            [query[1], train_array[position, 1]],
            color="0.30", alpha=0.55, linewidth=1)

# Окружность с радиусом, равным расстоянию до пятого соседа.
ax.add_patch(Circle(query, radius=distances[neighbor_positions[-1]],
                    fill=False, linestyle="--", linewidth=1.5, edgecolor="0.25"))
ax.scatter(train_array[neighbor_positions, 0], train_array[neighbor_positions, 1],
           s=135, facecolors="none", edgecolors="black", linewidths=1.6,
           label="Пять ближайших")
ax.scatter(query[0], query[1], marker="*", s=270, c="black", label="Новый объект", zorder=10)
ax.set_aspect("equal", adjustable="box")
ax.set(xlabel="Содержание алкоголя", ylabel="Количество флавоноидов",
       title="Какие наблюдения голосуют за новый объект?")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

В таблице пять соседей упорядочены по расстоянию до нового объекта. Пунктирная окружность проходит через пятого соседа: выбранные точки лежат внутри неё или на границе.

Каждый сосед голосует за свой класс. Долю голосов можно использовать как оценку вероятности, хотя это ещё не гарантирует её калиброванности. Далее сравним нашу реализацию с `KNeighborsClassifier` не только по отдельному прогнозу, но и **по качеству на validation**.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

my_knn = SimpleKNN(k=5).fit(X_wine_train, y_wine_train)
sklearn_knn = KNeighborsClassifier(n_neighbors=5, algorithm="brute").fit(
    X_wine_train, y_wine_train
)

# Одна validation-выборка и одни метрики для обеих реализаций.
my_predictions = my_knn.predict(X_wine_valid)
library_predictions = sklearn_knn.predict(X_wine_valid)

wine_quality = pd.DataFrame({
    "Реализация": ["SimpleKNN", "sklearn KNeighborsClassifier"],
    "Accuracy": [accuracy_score(y_wine_valid, my_predictions),
                 accuracy_score(y_wine_valid, library_predictions)],
    "Macro-F1": [f1_score(y_wine_valid, my_predictions, average="macro"),
                 f1_score(y_wine_valid, library_predictions, average="macro")],
})
wine_quality.round(3)

In [ ]:
# Удостоверимся также, что модели одинаково голосуют за классы.
# Сравниваем доли голосов для всех объектов validation, а не одного примера.
probability_difference = np.abs(
    my_knn.predict_proba(X_wine_valid)
    - sklearn_knn.predict_proba(X_wine_valid)
)
print("Различаются ли предсказанные метки:", np.any(my_predictions != library_predictions))
print("Максимальная разница долей классов:", probability_difference.max())

**Что показало сравнение?** Самостоятельная реализация и `KNeighborsClassifier` используют одно правило: вычисляют расстояния, находят ближайших обучающих соседей и агрегируют их ответы. На одинаковых данных и настройках получили одинаковую accuracy и macro-F1. У библиотечной реализации есть дополнительные возможности: способы взвешивания соседей, различные расстояния и алгоритмы ускоренного поиска, например KD-tree и BallTree.

Перейдём к реальной таблице, где помимо классификатора придётся продумать подготовку признаков.

---

## Часть II. Palmer Penguins: знакомство с задачей

**Palmer Penguins** — реальные наблюдения за взрослыми пингвинами трёх видов в районе архипелага Палмер у Антарктического полуострова. Данные собраны в 2007–2009 годах. На этот раз модель будет получать **не фотографии**, а табличные измерения и сведения о наблюдении.

![Три вида Palmer Penguins — рисунок Allison Horst](assets/lter_penguins.png)

*Иллюстрация: Allison Horst — Artwork by @allison_horst.*

![Три вида: иллюстрация к исследованию](assets/penguin_species_illustration.png)

### Как измеряют клюв

![Длина и глубина клюва — рисунок Allison Horst](assets/culmen_depth.png)

*Иллюстрация: Allison Horst — Artwork by @allison_horst.*

Длина клюва (`bill_length_mm`) и его глубина (`bill_depth_mm`) измеряются по разным направлениям. Это два самостоятельных числовых признака.

### Все исходные переменные

| Столбец | Что измеряет | Тип |
|---|---|---|
| `species` | Вид: **Adelie**, **Chinstrap** или **Gentoo** | Категориальная целевая переменная |
| `island` | Остров наблюдения: **Biscoe**, **Dream**, **Torgersen** | Номинальный признак |
| `bill_length_mm` | Длина клюва, мм | Числовой |
| `bill_depth_mm` | Глубина (высота) клюва, мм | Числовой |
| `flipper_length_mm` | Длина ласта, мм | Числовой |
| `body_mass_g` | Масса тела, граммы | Числовой |
| `sex` | Пол, если известен: `male` / `female` | Номинальный признак |
| `year` | Год обследования (2007–2009) | Временная характеристика наблюдения |

Цель исследования — предсказывать `species`. Год оставим для описания данных, но в базовый набор признаков не включим: он характеризует период сбора наблюдений, а не измеренную морфологию. Для первого эксперимента используем шесть признаков: четыре числовых и два категориальных.

### Загружаем локальную таблицу

In [ ]:
DATA_PATH = Path("penguins.csv")
penguins = pd.read_csv(DATA_PATH)
penguins.head(8)

In [ ]:
print("Строк и столбцов:", penguins.shape)
penguins.info()

`info()` показывает типы столбцов, число непустых значений и расход памяти. Оно позволяет заметить пропуски и ошибки чтения данных до построения модели.

In [ ]:
penguins.describe().T.round(2)

In [ ]:
# Для категорий также полезны частоты и число уникальных значений.
penguins.describe(include="object").T

### Сначала отделим test

Отложенная test-часть предназначена для одной итоговой проверки после выбора модели. Чтобы исследование и подбор гиперпараметров не влияли на неё, все последующие графики с целевой переменной, сравнения и настройку строим **на dev**. Метаданные исходной таблицы мы уже просмотрели; подробный EDA и выбор моделей проводим только на dev.

In [ ]:
numeric_features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
categorical_features = ["island", "sex"]
feature_names = numeric_features + categorical_features
X = penguins[feature_names]
y = penguins["species"]

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
dev = X_dev.assign(species=y_dev, year=penguins.loc[X_dev.index, "year"])
print("Dev:", X_dev.shape, "Test:", X_test.shape)

### EDA: доли видов и структура наблюдений

Доли классов важны и для выбора baseline, и для интерпретации macro-F1. Например, редкий класс может плохо распознаваться, даже если общая accuracy высока.

In [ ]:
dev["species"].value_counts().to_frame("Число наблюдений")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
order = ["Adelie", "Chinstrap", "Gentoo"]
sns.countplot(data=dev, x="species", order=order, color="#4C72B0", ax=axes[0])
axes[0].set(xlabel="Вид", ylabel="Число наблюдений", title="Баланс классов")
sns.countplot(data=dev, x="year", hue="species", hue_order=order, ax=axes[1])
axes[1].set(xlabel="Год", ylabel="Число наблюдений", title="Наблюдения по годам")
plt.tight_layout()
plt.show()

**Что показывает распределение видов?** Классы представлены неравномерно: Adelie встречается чаще Chinstrap. Поэтому высокая общая доля правильных ответов ещё не означает, что мы хорошо распознаём каждый вид. В дальнейшем сравним `accuracy` с macro-F1 и per-class recall.

### EDA: где находятся пропуски?

Посмотрим отдельно на количество отсутствующих значений и на **совместное расположение пропусков в строках**. Если несколько измерений отсутствуют у одних и тех же объектов, это может влиять на выбор способа заполнения.

In [ ]:
missing_counts = dev.isna().sum().sort_values(ascending=False)
missing_counts.to_frame("Число пропусков")

In [ ]:
rows_missing = dev[dev.isna().any(axis=1)].copy()
fig, ax = plt.subplots(figsize=(9, max(3, len(rows_missing) * 0.28 + 1)))
sns.heatmap(rows_missing[feature_names].isna(), cmap="Blues", cbar=False,
            linewidths=0.4, linecolor="white", ax=ax)
ax.set(xlabel="Признаки", ylabel="Строки с пропусками",
       title="Карта отсутствующих значений в dev")
plt.tight_layout()
plt.show()

**Как читать карту пропусков?** Каждая строка — наблюдение с хотя бы одним отсутствующим значением; каждый столбец — признак. Закрашенные клетки указывают на пропуски. Мы проверяем не только их количество, но и совпадают ли пропуски разных измерений в одних и тех же строках. Числовые пропуски и неизвестный пол потребуют разных стратегий заполнения.

Пропуски в `sex` мы **не удаляем и не игнорируем**. В дальнейшем для категориальных столбцов выберем отдельную стратегию заполнения и обучим её только на соответствующей тренировочной части. У числовых пропусков будет своя стратегия.

### EDA: распределения и возможные выбросы

Единицы измерения различаются: масса измеряется тысячами граммов, а размеры клюва — десятками миллиметров. Это станет существенным для kNN.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, feature in zip(axes.flat, numeric_features):
    sns.histplot(data=dev, x=feature, hue="species", multiple="layer",
                 bins=17, alpha=0.35, ax=ax, legend=False)
    ax.set(title=feature, ylabel="Число наблюдений")
plt.tight_layout()
plt.show()

**Наблюдение по распределениям.** Морфологические измерения видов различаются; особенно заметны особенности массы тела и размеров клюва. Но распределения перекрываются, поэтому классифицировать пингвинов по одному столбцу было бы ненадёжно. Кроме того, шкалы измерений неодинаковы: `body_mass_g` измеряется тысячами, а длины — десятками.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, feature in zip(axes.flat, numeric_features):
    sns.boxplot(data=dev, x="species", y=feature, order=order, color="#A4B7CF", ax=ax)
    ax.set(xlabel="Вид", ylabel=feature)
plt.tight_layout()
plt.show()

**Что дают boxplot?** Для каждого вида видны медиана, межквартильный диапазон и отдельные необычные значения признаков. График помогает сравнивать группы и замечать возможные выбросы; при этом необычное наблюдение не обязательно является ошибкой измерения.

### EDA: взаимосвязи признаков

На `pairplot` видны сразу попарные сочетания числовых признаков. По диагонали расположены их распределения, вне диагонали — диаграммы рассеяния. Это удобный первый способ оценить, какие признаки помогают разделять виды и где остаются пересечения.

In [ ]:
pair_data = dev[numeric_features + ["species"]].dropna()
g = sns.pairplot(pair_data, hue="species", corner=True,
                 plot_kws={"alpha": 0.65, "s": 22}, diag_kind="hist")
g.fig.suptitle("Palmer Penguins: попарные связи числовых признаков", y=1.02)
plt.show()

**Читаем pairplot.** На диагонали показаны распределения каждого измерения; вне диагонали — зависимости для пар признаков. Видны сочетания, по которым виды различаются лучше или хуже. Метод, использующий сразу несколько признаков, может опираться на их совместное положение, а не только на отдельный порог по одному измерению.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(dev[numeric_features].corr(), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax)
ax.set_title("Корреляции числовых признаков (dev)")
plt.tight_layout()
plt.show()

**Зачем нужна матрица корреляций?** Она компактно показывает линейные связи **между числовыми признаками**, а не их связь с видом. Например, масса тела и длина ласта могут возрастать вместе. Для kNN два сильно связанных признака могут усиливать вклад сходных различий в расстояние. Корреляция не показывает причинность и не решает сама по себе, какой признак следует исключить. В этом исследовании оставим измерения и проверим качество на CV.

**Промежуточные выводы.** Между видами существуют заметные различия в морфологических измерениях, но классы не идеально разделены. Некоторые измерения взаимосвязаны; для kNN это означает, что несколько коррелирующих признаков могут совместно усиливать вклад определённого типа различий в расстояние. По графикам нельзя заранее объявить одну модель лучшей — это предстоит проверить.

## Часть III. От baseline к корректной подготовке признаков

### Базовое правило до сложных моделей

Сначала оценим **DummyClassifier**, который всегда предсказывает самый частый вид в обучающей части. Этот классификатор вообще не использует признаки; зато он показывает нижний ориентир качества и напоминает, почему при нескольких классах одной accuracy бывает недостаточно.

Для сравнения будем использовать 5-fold `StratifiedKFold`: в каждом проходе доли видов в train и validation остаются примерно одинаковыми. `cross_val_score` каждый раз обучает **новый экземпляр модели** на тренировочных фолдах.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import f1_score, accuracy_score, classification_report, ConfusionMatrixDisplay

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = "f1_macro"
baseline = DummyClassifier(strategy="most_frequent")
baseline_folds = cross_val_score(baseline, X_dev, y_dev, cv=cv, scoring=scoring)
print("Macro-F1 по фолдам:", baseline_folds.round(3))
print("Средняя CV macro-F1:", round(baseline_folds.mean(), 3))

### Как работать с пропущенными значениями

Пропуск не всегда означает одно и то же. Измерение могли не выполнить, могли ошибиться при записи, а неизвестное значение категории может само по себе быть содержательным. На выбор стратегии влияет причина пропуска и тип столбца.

| Подход | Когда уместен | Что теряем или рискуем исказить |
|---|---|---|
| Удаление строк | Пропусков мало, удаление не меняет состав изучаемой совокупности | Потеря наблюдений, возможная систематическая ошибка |
| Среднее | Числовой признак без сильной асимметрии и выбросов | Чувствительность к экстремальным значениям |
| Медиана | Числовые признаки с выбросами или скошенным распределением | Теряется индивидуальная связь между признаками; одинаковая подстановка всем пропускам |
| Самая частая категория | Простой baseline для категориального признака | Усиливает долю распространённой категории; может скрывать особенности отсутствующих значений |
| Отдельная категория «Неизвестно» | Сам факт отсутствия значения может нести информацию | Отдельный уровень может быть шумным при малом числе пропусков |
| `KNNImputer` | Числовые столбцы, значения которых можно приблизить по похожим строкам | Зависимость от расстояния и масштаба, дополнительная вычислительная стоимость |

В нашем первом рабочем решении используем **медиану для числовых измерений и самый частый ответ для `sex`**. Все статистики заполнения должны определяться по train соответствующего фолда, а не по всей таблице перед CV.

### Почему нельзя заполнить значения «один раз навсегда»?

Даже замена пропуска на медиану требует её **оценивания по некоторой выборке**. Если предварительно вычислить медиану по всей таблице, а затем проводить CV, сведения о validation проникнут в этап подготовки train.

Сначала рассмотрим обычную ручную замену в одной маленькой таблице.

In [ ]:
toy = pd.DataFrame({"bill_length_mm": [37.1, np.nan, 41.0, 53.2]})
median_toy = toy["bill_length_mm"].median()
toy.assign(filled_manual=toy["bill_length_mm"].fillna(median_toy))

### `SimpleImputer`: обучаемый способ заполнения

`SimpleImputer` решает ту же задачу в стандартном интерфейсе sklearn.

- `fit(X_train)` **вычисляет** необходимые статистики по тренировочным данным и сохраняет их в `statistics_`;
- `transform(X_new)` применяет ранее найденные значения к новой таблице;
- `fit_transform(X_train)` выполняет обе операции последовательно на той же обучающей части.

Фактически этот объект обладает **обученным состоянием**, хотя не прогнозирует целевую переменную.

In [ ]:
from sklearn.impute import SimpleImputer

# Создаём обучаемое правило: заполнять пропуски медианой.
imputer_demo = SimpleImputer(strategy="median")
imputer_demo.fit(toy)

print("Медиана, сохранённая при fit:", imputer_demo.statistics_)

# transform возвращает массив NumPy; отдельно делаем из него привычную таблицу.
toy_filled_array = imputer_demo.transform(toy)
toy_filled_table = pd.DataFrame(toy_filled_array, columns=toy.columns)
toy_filled_table

### Состояние преобразователя на настоящей таблице

Теперь вычислим медианы признаков на dev и посмотрим, что запомнил имьютер. Это отдельная демонстрация работы `fit`. При кросс-валидации каждый train-фолд будет иметь **собственный** обученный имьютер: значения, полученные по всему dev, не используются для подготовки validation-строк.

In [ ]:
num_imputer_demo = SimpleImputer(strategy="median")
num_imputer_demo.fit(X_dev[numeric_features])
pd.Series(num_imputer_demo.statistics_, index=numeric_features,
          name="Медиана по dev")

In [ ]:
# Покажем только строки с числовыми пропусками.
missing_numeric = X_dev[X_dev[numeric_features].isna().any(axis=1)]
num_filled = pd.DataFrame(
    num_imputer_demo.transform(missing_numeric[numeric_features]),
    index=missing_numeric.index, columns=numeric_features
)
num_filled.head()

### Что происходит с категориальным `sex`?

Числовая медиана не подходит для значений `male` и `female`. Можно заполнить их наиболее распространённой меткой либо заранее завести категорию «не указан». Для первого baseline выберем **most_frequent** и посмотрим, какое значение запоминается.

In [ ]:
sex_imputer_demo = SimpleImputer(strategy="most_frequent")
sex_imputer_demo.fit(X_dev[["sex"]])
print("Метка для заполнения:", sex_imputer_demo.statistics_[0])
sex_missing = X_dev.loc[X_dev["sex"].isna(), ["sex"]]
print("Пропусков sex на dev:", len(sex_missing))
if len(sex_missing):
    pd.DataFrame(sex_imputer_demo.transform(sex_missing), columns=["sex"],
                 index=sex_missing.index).head()
else:
    print("В этой части выборки пропусков нет")

### Масштабирование: StandardScaler и MinMaxScaler

У kNN порядок соседей зависит от шкал признаков. Поэтому после заполнения пропусков часто выполняют масштабирование.

**StandardScaler:** вычитает тренировочное среднее и делит на тренировочное стандартное отклонение:

$$z=\frac{x-\mu_{train}}{s_{train}}.$$

**MinMaxScaler:** использует тренировочные минимум и максимум:

$$z=\frac{x-\min_{train}}{\max_{train}-\min_{train}}.$$

Если новый объект выходит за пределы тренировочного диапазона, его значение после MinMaxScaler может быть меньше 0 или больше 1. В обоих случаях параметры преобразования определяются только по train.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaling_demo = pd.DataFrame({"mass": [3000, 4000, 5000, 6000]})
standard_demo = StandardScaler().fit(scaling_demo)
minmax_demo = MinMaxScaler().fit(scaling_demo)
pd.DataFrame({
    "исходное": scaling_demo["mass"],
    "StandardScaler": standard_demo.transform(scaling_demo).ravel(),
    "MinMaxScaler": minmax_demo.transform(scaling_demo).ravel(),
}).round(3)

In [ ]:
print("StandardScaler.mean_:", standard_demo.mean_)
print("StandardScaler.scale_:", standard_demo.scale_)
print("MinMaxScaler.data_min_:", minmax_demo.data_min_)
print("MinMaxScaler.data_max_:", minmax_demo.data_max_)

Две важные особенности: `fit` запоминает параметры по тренировочной таблице; `transform` использует их при обработке любого последующего объекта. Так же работают многие другие обучаемые преобразователи sklearn.

## Часть IV. `Pipeline`: объединяем преобразования и модель

До этого мы применяли методы вручную: отдельно обучали имьютер, отдельно масштабировали таблицу, отдельно вызывали классификатор. Такой код легко сделать неправильным при кросс-валидации: преобразователи могут случайно увидеть не свою часть данных.

`Pipeline` описывает **последовательность шагов**, у которых общий интерфейс `fit` и `predict`. Для наших числовых признаков:

$$\text{пропуски}\longrightarrow\text{масштабирование}\longrightarrow\text{kNN}.$$

`Pipeline` — это не ещё один алгоритм классификации. Это способ объединить преобразователи и модель в один объект, который можно обучать и оценивать как единое целое.

### Анатомия `Pipeline`

Каждый шаг имеет имя и объект. Промежуточные шаги должны уметь преобразовывать данные (`fit` и `transform`), а последний шаг — решать конечную задачу, например классификацию (`fit` и `predict`).

При вызове **`pipe.fit(X_train, y_train)`** происходит следующее:

1. `imputer.fit_transform(X_train)` — вычисляет медианы и заполняет пропуски;
2. `scaler.fit_transform(...)` — определяет параметры масштаба и преобразует матрицу;
3. `knn.fit(...)` — сохраняет уже подготовленные тренировочные объекты и их классы.

При **`pipe.predict(X_new)`** первые два шага вызывают **только `transform`**, после чего обученный kNN строит прогноз. Никаких новых медиан или средних по `X_new` не вычисляется.

In [ ]:
from sklearn.pipeline import Pipeline

numeric_knn = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])

numeric_knn

### Почему шаги объединяют до кросс-валидации

Если передать всю цепочку в `cross_val_score`, sklearn для каждого фолда создаст новую, ещё не обученную цепочку и выполнит её `fit` только на тренировочных строках. Это относится и к медианам, и к стандартным отклонениям, и к самому классификатору.

Именно поэтому корректный pipeline удобен не только как организация кода, но и как защита от утечки между train и validation.

In [ ]:
numeric_knn_scores = cross_val_score(
    numeric_knn, X_dev[numeric_features], y_dev, cv=cv, scoring=scoring
)
print("kNN с медианой и StandardScaler:", numeric_knn_scores.round(3))
print("CV macro-F1:", round(numeric_knn_scores.mean(), 3))

### Сравнение: без scaling, StandardScaler и MinMaxScaler

Теперь сравним три цепочки с **одинаковым kNN**, одинаковыми фолдами и одинаковым заполнением пропусков. Меняется только геометрия после преобразования числовых признаков.

**Зачем `clone`?** Функция `sklearn.base.clone` создаёт новый объект estimator с **теми же гиперпараметрами, но без обученного состояния**. Это удобно, если мы хотим независимо обучить одинаковые схемы и не переносить случайно запомненные статистики из одного эксперимента в другой. Внутри `cross_val_score` и `GridSearchCV` sklearn также создаёт независимые экземпляры модели для разных фолдов и настроек.

In [ ]:
from sklearn.base import clone  # Новая необученная копия с теми же настройками.

numeric_versions = {
    "Без масштабирования": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("knn", KNeighborsClassifier(n_neighbors=5)),
    ]),
    "StandardScaler": clone(numeric_knn),
    "MinMaxScaler": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MinMaxScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5)),
    ]),
}
comparison_numeric = {}
for name, model in numeric_versions.items():
    comparison_numeric[name] = cross_val_score(
        model, X_dev[numeric_features], y_dev, cv=cv, scoring=scoring
    )
scaling_scores = pd.DataFrame(comparison_numeric)
scaling_scores.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
scaling_scores.mean().sort_values().plot.barh(ax=ax, xerr=scaling_scores.std(),
                                               capsize=4)
ax.set(xlabel="Средняя CV macro-F1 (± SD фолдов)", ylabel="",
       title="Один kNN, разные варианты подготовки признаков", xlim=(0, 1))
plt.tight_layout()
plt.show()

**Интерпретация.** Масштабирование изменяет сами расстояния и набор ближайших соседей; поэтому выигрыш нельзя объяснить одной лишь скоростью работы алгоритма. Различия между StandardScaler и MinMaxScaler зависят от данных: оба метода разумно рассматривать как варианты preprocessing и проверять в одной схеме CV.

### Наглядный эффект на конкретном соседе

Для одного train/validation-разбиения сравним идентификаторы ближайших обучающих строк до и после масштабирования. Здесь нас интересуют **состав и порядок соседей**, а не ещё одна оценка обобщающей способности.

In [ ]:
train_idx, valid_idx = next(cv.split(X_dev, y_dev))
X_fold_train = X_dev.iloc[train_idx][numeric_features]
X_fold_valid = X_dev.iloc[valid_idx][numeric_features]

def nearest_five(preprocessing):
    X_train_ready = preprocessing.fit_transform(X_fold_train)
    X_valid_ready = preprocessing.transform(X_fold_valid)
    d = np.linalg.norm(X_train_ready - X_valid_ready[0], axis=1)
    return X_fold_train.index[np.argsort(d)[:5]].tolist()

raw_neighbors = nearest_five(SimpleImputer(strategy="median"))
scaled_neighbors = nearest_five(Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
]))
print("До scaling:", raw_neighbors)
print("После scaling:", scaled_neighbors)

## Часть V. Категориальные признаки и `ColumnTransformer`

Пока kNN использовал только четыре числовых признака. Но у пингвинов есть ещё `island` и `sex`.

**Нельзя просто заменить острова числами 0, 1, 2:** евклидово расстояние начнёт приписывать этим кодам количественный смысл. Для номинальных признаков рассмотрим one-hot encoding — отдельный индикатор для каждого известного значения.

In [ ]:
# Как часто встречаются категории в dev?
pd.crosstab(dev["species"], dev["island"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=dev, x="island", hue="species", ax=axes[0])
axes[0].set(title="Остров наблюдения", xlabel="Остров", ylabel="Число объектов")
sns.countplot(data=dev, x="sex", hue="species", ax=axes[1])
axes[1].set(title="Пол (пропуски не показаны)", xlabel="Пол", ylabel="Число объектов")
plt.tight_layout()
plt.show()

### `OneHotEncoder`: чему он обучается и какие варианты существуют

В `fit` кодировщик определяет категории каждого столбца и запоминает их в `categories_`. При `transform` он заменяет категории индикаторами. В отличие от `LabelEncoder`, который применяется к целевой переменной, `OneHotEncoder` предназначен для категориальных **признаков**.

Для значений, которые не встречались в `fit`, возможны разные правила:

- `handle_unknown="error"` (по умолчанию) — остановиться с ошибкой;
- `handle_unknown="ignore"` — получить нули в индикаторах неизвестной категории;
- `handle_unknown="infrequent_if_exist"` — отправить новую категорию в общую группу редких, **если такая группа была создана** с помощью `min_frequency` или `max_categories`.

Сначала заполним отсутствующие значения `sex` самой частой известной категорией. Это выбранное правило для данного примера, а не единственный возможный способ работы с пропусками.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_demo = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
categorical_demo.fit(X_dev[categorical_features])
encoder_demo = categorical_demo.named_steps["onehot"]
print("Категории острова и пола:", encoder_demo.categories_)
print("Имена новых столбцов:", encoder_demo.get_feature_names_out(categorical_features))

In [ ]:
demo_categories = X_dev[categorical_features].head(5)
encoded_categories = categorical_demo.transform(demo_categories)
pd.DataFrame(encoded_categories,
             columns=encoder_demo.get_feature_names_out(categorical_features),
             index=demo_categories.index)

### Неизвестные категории

Представим, что модель встретила остров, отсутствовавший при `fit`. Здесь `handle_unknown="ignore"` позволяет продолжать работу. Но это не означает, что модель узнала, насколько новый остров похож на известные.

In [ ]:
new_row = pd.DataFrame({"island": ["Unknown island"], "sex": ["female"]})
pd.DataFrame(categorical_demo.transform(new_row),
             columns=encoder_demo.get_feature_names_out(categorical_features))

### Зачем нужен `ColumnTransformer`

С одним типом столбцов достаточно обычного `Pipeline`: преобразования выполняются **друг за другом** над всей полученной таблицей. Но в Penguins есть и измерения, и названия категорий. Одну и ту же операцию к ним применять нельзя: например, медиана и `StandardScaler` не имеют смысла для строки `female`.

`ColumnTransformer` решает **другую задачу**: выбирает группы исходных столбцов, применяет к каждой группе свою ветвь преобразований и соединяет результаты **по горизонтали**.

| Объект | Что происходит при `fit` | Что происходит при `transform` |
|---|---|---|
| Числовой `Pipeline` | Вычисляет медианы и параметры масштаба | Заполняет и масштабирует числа |
| Категориальный `Pipeline` | Находит типичные категории и список one-hot столбцов | Заполняет и кодирует категории |
| `ColumnTransformer` | Обучает каждую ветвь **на выбранных столбцах** | Применяет ветви и объединяет результат |

Таким образом, `Pipeline` организует **последовательность операций**, а `ColumnTransformer` — **параллельную обработку разных столбцов**. Эти объекты естественно вкладываются друг в друга.

### Сначала строим две ветви

In [ ]:
numeric_branch = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

In [ ]:
categorical_branch = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

Каждый шаг имеет имя; порядок внутри ветви важен. Например, числовое масштабирование должно получать значения без пропусков. Категориальный кодировщик получает заполненные строковые категории.

### Интерфейс `ColumnTransformer`: имя, преобразователь, столбцы

При создании объекта каждая ветвь задаётся тройкой:

`("имя", преобразователь, ["список", "столбцов"])`.

Имя понадобится для доступа к обученной ветви через `named_transformers_`. Преобразователь — это, например, `Pipeline` или отдельный `OneHotEncoder`. Последний элемент говорит, **к каким исходным столбцам** применять преобразование.

Оставшиеся столбцы по умолчанию отбрасываются (`remainder="drop"`); при `remainder="passthrough"` их можно передать дальше без обработки. Порядок итоговых столбцов определяется порядком перечисленных ветвей.

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", numeric_branch, numeric_features),
    ("cat", categorical_branch, categorical_features),
])
preprocessor

### Как выглядят данные после `ColumnTransformer`

Обучим отдельный экземпляр на dev, чтобы посмотреть результат преобразования. Он пригодится для иллюстрации, а в настоящей CV каждая ветвь будет обучаться заново на соответствующей тренировочной части.

Для небольшого набора Penguins создаём **плотную** one-hot матрицу с `sparse_output=False`. Это позволяет показать итог как обычную таблицу pandas; для больших таблиц разреженное представление часто экономнее по памяти.

In [ ]:
# Берём свежий необученный объект, чтобы показать его состояние после fit.
preprocessor_demo = clone(preprocessor)
X_dev_ready = preprocessor_demo.fit_transform(X_dev)

print("Исходная таблица:", X_dev.shape)
print("После подготовки:", X_dev_ready.shape)
print("Новые столбцы:", list(preprocessor_demo.get_feature_names_out()))

In [ ]:
# В этой учебной задаче ColumnTransformer возвращает плотный массив NumPy.
# Преобразуем его в DataFrame, чтобы увидеть названия итоговых столбцов.
ready_columns = preprocessor_demo.get_feature_names_out()
ready_table = pd.DataFrame(X_dev_ready, columns=ready_columns, index=X_dev.index)
ready_table.head().round(2)

### Полный `Pipeline`: одна цепочка с исходной таблицей на входе

Теперь соединим подготовку и классификатор:

$$\underbrace{X_{raw}}_{\text{числа и категории}}
\xrightarrow{\text{ColumnTransformer}}
\underbrace{X_{ready}}_{\text{числовая матрица}}
\xrightarrow{\text{kNN}}\hat y.$$

При `fit` подготовка признаков и обучение классификатора выполняются последовательно. При `predict` к новым исходным строкам применяются **уже запомненные** преобразования, а затем обученный kNN.

In [ ]:
# Значение n_neighbors=5 здесь стартовое: оно позволяет обучать Pipeline до настройки.
# Позже GridSearchCV перебирает и ПЕРЕОПРЕДЕЛЯЕТ этот параметр через model__n_neighbors.
full_knn = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", KNeighborsClassifier(n_neighbors=5)),
])
full_knn

In [ ]:
full_cv_scores = cross_val_score(full_knn, X_dev, y_dev, cv=cv, scoring=scoring)
print("CV macro-F1:", full_cv_scores.round(3))
print("Среднее:", round(full_cv_scores.mean(), 3))

### Пример утечки при подготовке данных

Допустим, мы хотим оценить модель на одном из CV-фолдов. Чтобы подготовить его training-часть, нужно заполнить пропуски и вычислить масштаб числовых признаков. **Значения из validation не должны участвовать в этом вычислении.**

Покажем две процедуры на одном разбиении:

1. Правильно: обучаем числовую ветвь по train текущего фолда.
2. Неправильно: обучаем её по всему dev, куда входят и объекты будущей validation.

Сравним запомненные значения `StandardScaler.mean_`. Это пример самой утечки; большой разницы в метрике для её обнаружения не требуется.

In [ ]:
# Берём первый train/validation-фолд из нашей 5-fold CV.
train_ids, valid_ids = next(cv.split(X_dev, y_dev))
X_fold_train = X_dev.iloc[train_ids]
X_fold_valid = X_dev.iloc[valid_ids]

# В обоих случаях используем одинаковую схему: median -> StandardScaler.
# Корректно: статистики вычисляются ТОЛЬКО по обучающим строкам этого фолда.
correct_prep = clone(numeric_branch)
correct_prep.fit(X_fold_train[numeric_features])
correct_mean = correct_prep.named_steps["scaler"].mean_

# Некорректно: fit видит также строки будущей validation-части.
leaking_prep = clone(numeric_branch)
leaking_prep.fit(X_dev[numeric_features])
leaking_mean = leaking_prep.named_steps["scaler"].mean_

# Разница — не «качество» моделей, а именно утечка в обученном состоянии.
pd.DataFrame({
    "Только train-фолд": correct_mean,
    "Весь dev (с validation)": leaking_mean,
    "Различие": leaking_mean - correct_mean,
}, index=numeric_features).round(3)

Даже если различия статистик невелики, оценивание должно воспроизводить реальную ситуацию: при получении нового объекта его признаки не использовались для обучения преобразований. Передача полного pipeline в `cross_val_score` или `GridSearchCV` обеспечивает нужный порядок автоматически.

## Часть VI. Подбираем гиперпараметры kNN и сравниваем с логистической регрессией

До сих пор для примеров использовали `n_neighbors=5`. Теперь используем знакомый `GridSearchCV`, чтобы **выбрать настройки**, а не задавать их произвольно.

`GridSearchCV` получает:

1. `estimator` — всю цепочку, включая подготовку признаков;
2. `param_grid` — комбинации проверяемых гиперпараметров;
3. `cv` — разбиения, одинаковые для всех кандидатов;
4. `scoring` — метрику сравнения, здесь macro-F1.

Для каждой комбинации параметров sklearn создаёт новую цепочку и оценивает её на всех фолдах. Параметр `model__n_neighbors` относится к вложенному шагу `model`; он заменяет стартовое `n_neighbors=5`. После выбора при `refit=True` (значение по умолчанию) лучшая цепочка обучается **на всём dev**, и её можно получить как `best_estimator_`.

### Сетка для kNN

Проверим как нечётные, так и чётные значения числа соседей: ничья при чётном `k` возможна, но алгоритм должен уметь её разрешать. Дополнительно сравним равные голоса с весами по расстоянию, а также евклидово и манхэттенское расстояния.

Двойное подчёркивание в `model__n_neighbors` означает: «параметр `n_neighbors` шага `model` внутри Pipeline».

In [ ]:
from sklearn.model_selection import GridSearchCV

# Число соседей, способ голосования и расстояние: три гиперпараметра kNN.
knn_param_grid = {
    "model__n_neighbors": [1, 2, 3, 4, 5, 7, 9, 11, 15, 21],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"],
}
print("Всего комбинаций:", 10 * 2 * 2)

# Для каждой комбинации и каждого фолда обучается свой Pipeline.
knn_search = GridSearchCV(
    full_knn, knn_param_grid, cv=cv, scoring=scoring,
    return_train_score=True, n_jobs=1, refit=True,
)
knn_search.fit(X_dev, y_dev)

In [ ]:
print("Лучшие настройки kNN:", knn_search.best_params_)
print("Лучшая CV macro-F1:", round(knn_search.best_score_, 4))
print("Число соседей в переобученной лучшей модели:",
      knn_search.best_estimator_.named_steps["model"].n_neighbors)

# Каждая строка таблицы соответствует отдельной комбинации гиперпараметров.
knn_search_table = pd.DataFrame(knn_search.cv_results_)
selected_columns = [
    "param_model__n_neighbors", "param_model__weights", "param_model__metric",
    "mean_test_score", "std_test_score", "rank_test_score",
]
knn_search_table[selected_columns].sort_values(
    ["rank_test_score", "param_model__n_neighbors"]
).head(12).round(4)

### Что означают одинаковые значения в таблице поиска?

Разные настройки kNN **могут действительно получить одну и ту же оценку** на небольшой выборке: многие комбинации приводят к одинаковым предсказаниям на validation-частях.

В таблице `cv_results_` сохранены все 40 комбинаций, в том числе различные значения `n_neighbors`. По параметру `rank_test_score` можно увидеть совпадения качества. `best_estimator_` имеет именно те значения, которые указаны в `best_params_`.

Небольшое различие средних CV-показателей не гарантирует устойчивое превосходство одной настройки. Мы выбираем разумного кандидата и затем оцениваем его на отложенном test.

In [ ]:
# Покажем зависимость CV-качества от k для разных способов голосования.
plot_knn_results = knn_search_table.copy()
plot_knn_results["k"] = plot_knn_results["param_model__n_neighbors"].astype(int)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, metric in zip(axes, ["euclidean", "manhattan"]):
    subset = plot_knn_results[plot_knn_results["param_model__metric"] == metric]
    sns.lineplot(data=subset, x="k", y="mean_test_score", hue="param_model__weights",
                 marker="o", ax=ax)
    ax.set(xlabel="Число соседей k", ylabel="Средняя CV macro-F1",
           title=f"Метрика: {metric}")
    ax.set_xticks([1, 2, 3, 4, 5, 7, 9, 11, 15, 21])
plt.tight_layout()
plt.show()

### LogisticRegression: другая модель, тот же эксперимент

Логистическая регрессия не ищет соседей: она строит совместную многоклассовую модель. Но **входная таблица, подготовка признаков, CV и метрика** должны быть сопоставимыми с kNN.

Мы меняем только последний шаг Pipeline и подбираем силу регуляризации через `C` (`C` обратно связан с силой L2-штрафа).

In [ ]:
from sklearn.linear_model import LogisticRegression

# Меняем финальную модель, но сохраняем подготовку и CV-протокол.
logreg_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", LogisticRegression(max_iter=3000)),
])

# C = 1 / сила регуляризации: больше C -> слабее штраф.
logreg_grid = {"model__C": [0.01, 0.1, 1, 10, 100]}
logreg_search = GridSearchCV(
    logreg_pipeline, logreg_grid, cv=cv, scoring=scoring,
    n_jobs=1, refit=True,
)
logreg_search.fit(X_dev, y_dev)

In [ ]:
print("Лучшее C:", logreg_search.best_params_)
print("Лучшая CV macro-F1:", round(logreg_search.best_score_, 3))

pd.DataFrame(logreg_search.cv_results_)[[
    "param_model__C", "mean_test_score", "std_test_score"
]].sort_values("mean_test_score", ascending=False).round(3)

### Сравним результаты на dev

Baseline, kNN и логистическую регрессию оценивали на одном dev и одних фолдах. В таблице ниже сравниваем именно результаты CV, не используя отложенный test.

In [ ]:
cv_comparison_full = pd.DataFrame({
    "Модель": ["Dummy (самый частый класс)", "kNN", "LogisticRegression"],
    "Средняя CV macro-F1": [baseline_folds.mean(),
                           knn_search.best_score_, logreg_search.best_score_],
})
cv_comparison_full.round(3)

### Out-of-fold прогнозы: разбираем ошибки на dev

Средняя CV macro-F1 описывает качество модели одним числом. Но для матрицы ошибок нам нужны **предсказания отдельных объектов**. Нельзя просто вызвать `predict(X_dev)` у модели, обученной на всём dev: такие прогнозы получены на обучающих наблюдениях.

`cross_val_predict` организует другой процесс:

1. Делит dev на те же пять фолдов.
2. На каждом шаге обучает полную цепочку на четырёх фолдах.
3. Получает предсказания для пятого, не виденного этой копией модели фолда.
4. Собирает предсказания обратно в исходном порядке строк.

Так каждый объект dev имеет **out-of-fold (OOF)** прогноз — от модели, которая не обучалась на этом объекте.

Гиперпараметры мы уже подобрали на dev. Поэтому OOF здесь нужен **для разбора ошибок выбранной настройки**, а не как новая полностью независимая оценка выбора гиперпараметров. Финальный test по-прежнему остаётся отдельно.

Для наглядности посмотрим размеры train и validation каждого фолда. Каждая строка dev оказывается в validation ровно один раз.

In [ ]:
fold_summary = []
for fold_number, (train_ids, valid_ids) in enumerate(cv.split(X_dev, y_dev), start=1):
    fold_summary.append({
        "Фолд": fold_number,
        "Train объектов": len(train_ids),
        "Validation объектов": len(valid_ids),
    })
pd.DataFrame(fold_summary)

In [ ]:
class_order = ["Adelie", "Chinstrap", "Gentoo"]

# Каждый прогноз создаётся моделью, которая НЕ обучалась на данной строке dev.
# На входе — выбранная конфигурация Pipeline; внутри CV она переобучается по фолдам.
knn_oof = cross_val_predict(knn_search.best_estimator_, X_dev, y_dev, cv=cv)
print("OOF accuracy:", round(accuracy_score(y_dev, knn_oof), 3))
print("OOF macro-F1:", round(f1_score(y_dev, knn_oof, average="macro"), 3))

ConfusionMatrixDisplay.from_predictions(
    y_dev, knn_oof, labels=class_order, cmap="Blues", colorbar=False
)
plt.title("kNN: ошибки на отложенных CV-фолдах (OOF)")
plt.show()

In [ ]:
print(classification_report(
    y_dev, knn_oof, labels=class_order, digits=3, zero_division=0
))

В `classification_report` колонка `support` — число истинных объектов каждого класса. `recall` отвечает на вопрос, какую долю объектов данного вида модель распознала; `precision` — какая доля объектов с предсказанным названием вида действительно имела этот вид. `macro avg` равновесно учитывает классы.

### Диагностика логистической регрессии

Теперь повторим процедуру OOF для логистической регрессии. **Фолды не меняются**: обе модели делают предсказания на тех же объектах, что позволяет сопоставлять не только среднее качество, но и характер ошибок. Итоговая таблица показывает, какие виды распознаются хуже.

In [ ]:
logreg_oof = cross_val_predict(logreg_search.best_estimator_, X_dev, y_dev, cv=cv)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, pred, name in zip(axes, [knn_oof, logreg_oof], ["kNN", "LogisticRegression"]):
    ConfusionMatrixDisplay.from_predictions(
        y_dev, pred, labels=class_order, cmap="Blues", colorbar=False, ax=ax
    )
    ax.set_title(f"OOF: {name}")
plt.tight_layout()
plt.show()

In [ ]:
print("LogisticRegression: отчет по классам")
print(classification_report(
    y_dev, logreg_oof, labels=class_order, digits=3, zero_division=0
))

### Разберём конкретные ошибки

В таблицу ошибок добавим истинный и предсказанный виды. Так можно увидеть, какие группы наблюдений путаются, и не потерять связь с исходными измерениями.

In [ ]:
oof_errors = X_dev.copy()
oof_errors["истинный вид"] = y_dev
oof_errors["kNN"] = knn_oof
oof_errors["LogReg"] = logreg_oof
errors_any = oof_errors[
    (oof_errors["истинный вид"] != oof_errors["kNN"])
    | (oof_errors["истинный вид"] != oof_errors["LogReg"])
]
print("Строк, где ошиблась хотя бы одна модель:", len(errors_any))
errors_any.head(12)

**Что узнали из первого цикла?** На этой выборке морфологические измерения несут сильный сигнал для распознавания вида. Из `classification_report` видно, какие виды определяются точнее; таблица ошибочных строк помогает увидеть, где модели расходятся.

Однако мы добавили к измерениям **остров наблюдения**. Проверим, насколько он связан с видом и нужен ли он в предметной постановке: хотим ли мы классифицировать пингвина по его собственным характеристикам или разрешаем использовать также место наблюдения?

## Часть VII. Остров наблюдения: полезный признак или не та информация?

Остров наблюдения сильно связан с видом: Gentoo в наборе встречаются на Biscoe, Chinstrap — на Dream, а Adelie представлены на всех трёх островах. Это не утечка само по себе: остров мог быть известен при предсказании. Однако качество такой модели частично отражает **географическую структуру конкретной выборки**.

Поставим более предметный вопрос: можно ли определять вид по измерениям и полу пингвина, **не используя место наблюдения**? Сначала посмотрим, насколько распределение видов различается между островами, затем заново сравним модели без `island`.

In [ ]:
island_table = pd.crosstab(dev["species"], dev["island"])
island_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(island_table, annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False)
axes[0].set(xlabel="Остров", ylabel="Вид", title="Количество наблюдений")
island_table.div(island_table.sum(axis=1), axis=0).plot(
    kind="bar", stacked=True, ax=axes[1]
)
axes[1].set(xlabel="Вид", ylabel="Доля наблюдений", title="Доли островов внутри каждого вида")
axes[1].legend(title="Остров", bbox_to_anchor=(1.0, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Новая постановка: исключаем остров и повторяем исследование

Теперь остаются четыре морфологических измерения и `sex` — категориальный признак самого пингвина. Мы снова создадим числовую и категориальную ветви, соберём для каждой модели собственный `Pipeline`, подберём настройки на той же dev-выборке и разберём ошибки.

**Мы не переиспользуем обученные преобразователи предыдущей постановки:** у новых моделей другой набор исходных столбцов. В коде ниже создадим все компоненты заново, чтобы было видно, из чего они состоят.

In [ ]:
# Вход новой модели: только измерения и пол, без места наблюдения.
features_no_island = numeric_features + ["sex"]
X_dev_no_island = X_dev[features_no_island]
X_test_no_island = X_test[features_no_island]

X_dev_no_island.head()

### Заново задаём обработку числовых и категориальных столбцов

Числовые измерения сначала заполняем медианой и затем стандартизируем. Для `sex` заполняем пропуски самой частой категорией и выполняем one-hot encoding. Новая таблица будет иметь пять исходных столбцов, а количество столбцов после кодирования изменится.

In [ ]:
# Отдельные НОВЫЕ ветви для новой постановки, без копирования fitted объектов.
numeric_no_island = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_no_island = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor_no_island = ColumnTransformer([
    ("num", numeric_no_island, numeric_features),
    ("cat", categorical_no_island, ["sex"]),
])
preprocessor_no_island

### Две модели — два независимых полных pipeline

Для kNN и логистической регрессии подготовка признаков одинакова **по смыслу**, но обученные экземпляры должны быть независимыми. Явно создадим вторую схему preprocessing для логистической регрессии. В обоих случаях `GridSearchCV` будет обучать новый экземпляр всей цепочки на каждом фолде.

In [ ]:
# Первая полная цепочка — kNN.
knn_no_island = Pipeline([
    ("preprocess", preprocessor_no_island),
    ("model", KNeighborsClassifier(n_neighbors=5)),
])

# Для логистической регрессии заново задаём свои преобразователи.
logreg_numeric_no_island = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
logreg_categorical_no_island = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
logreg_preprocessor_no_island = ColumnTransformer([
    ("num", logreg_numeric_no_island, numeric_features),
    ("cat", logreg_categorical_no_island, ["sex"]),
])
logreg_no_island = Pipeline([
    ("preprocess", logreg_preprocessor_no_island),
    ("model", LogisticRegression(max_iter=3000)),
])
print("kNN steps:", list(knn_no_island.named_steps))
print("LogReg steps:", list(logreg_no_island.named_steps))

### Повторяем подбор kNN

В новой постановке проверяем **те же значения числа соседей, расстояния и веса**, что и в первой. Это позволяет отделить эффект удаления признака `island` от эффекта выбора другой сетки настройки. На каждом фолде заново обучаются и подготовка, и классификатор.

In [ ]:
# GridSearchCV сам перебирает значения k, не ограничиваясь стартовым n_neighbors=5.
knn_no_island_search = GridSearchCV(
    knn_no_island, knn_param_grid, scoring=scoring, cv=cv, n_jobs=1, refit=True
)
knn_no_island_search.fit(X_dev_no_island, y_dev)
print("Лучшие параметры:", knn_no_island_search.best_params_)
print("CV macro-F1:", round(knn_no_island_search.best_score_, 4))

### Повторяем подбор логистической регрессии

Для неё на той же CV снова перебираем коэффициент `C`. Числовая и категориальная ветви также обучаются отдельно на тренировочной части каждого фолда.

In [ ]:
logreg_no_island_search = GridSearchCV(
    logreg_no_island, logreg_grid, scoring=scoring, cv=cv, n_jobs=1, refit=True
)
logreg_no_island_search.fit(X_dev_no_island, y_dev)
print("Лучшее C:", logreg_no_island_search.best_params_)
print("CV macro-F1:", round(logreg_no_island_search.best_score_, 4))

### Сравниваем две постановки на dev

Первая модель могла использовать место наблюдения, вторая опирается только на собственные признаки пингвина. По одинаковой схеме CV посмотрим, сколько качества мы теряем или выигрываем, исключая `island`. Здесь важны не просто две близкие доли правильных ответов, а то, **какой предметный вопрос** каждая модель решает.

In [ ]:
ablation_comparison = pd.DataFrame({
    "Модель": ["kNN", "LogisticRegression"],
    "С островом": [knn_search.best_score_, logreg_search.best_score_],
    "Без острова": [knn_no_island_search.best_score_,
                     logreg_no_island_search.best_score_],
})
# Разница в процентных пунктах удобнее неинформативного столбчатого графика 0..1.
ablation_comparison["Изменение, п. п."] = (
    ablation_comparison["Без острова"] - ablation_comparison["С островом"]
) * 100
ablation_comparison.round(3)

In [ ]:
# Распределение видов по островам: строки суммируются до 100 %.
# Такая таблица объясняет, почему island был информативен для классификатора.
island_species_share = pd.crosstab(
    dev["island"], dev["species"], normalize="index"
) * 100
island_species_share.round(1)

### Диагностика моделей без острова по отложенным фолдам

Снова применим `cross_val_predict`: в каждом фолде создаётся и обучается новая цепочка с выбранными гиперпараметрами, а предсказания записываются **только для validation-строк**. Повторим матрицы ошибок и отчёты по классам для обеих моделей.

Настройки уже выбирались по dev, поэтому эти OOF-прогнозы полезны прежде всего для **диагностики ошибок**. Об итоговом качестве будем судить по test, который ещё не использовали.

In [ ]:
knn_no_island_oof = cross_val_predict(
    knn_no_island_search.best_estimator_, X_dev_no_island, y_dev, cv=cv
)
logreg_no_island_oof = cross_val_predict(
    logreg_no_island_search.best_estimator_, X_dev_no_island, y_dev, cv=cv
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, prediction, name in zip(
    axes,
    [knn_no_island_oof, logreg_no_island_oof],
    ["kNN без island", "LogisticRegression без island"],
):
    # В каждом фолде train и validation различаются.
    ConfusionMatrixDisplay.from_predictions(
        y_dev, prediction, labels=class_order,
        cmap="Blues", colorbar=False, ax=ax
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
print("kNN без island")
print(classification_report(y_dev, knn_no_island_oof,
                            labels=class_order, zero_division=0, digits=3))

In [ ]:
print("LogisticRegression без island")
print(classification_report(y_dev, logreg_no_island_oof,
                            labels=class_order, zero_division=0, digits=3))

**Вывод по эксперименту без острова.** Сравнение на одинаковых фолдах показывает, насколько информативны морфологические измерения и пол сами по себе. Таблица изменения CV macro-F1 позволяет количественно оценить эффект исключения `island`, а матрицы ошибок — проверить, какие виды стали чаще путаться. Теперь можно оценить обе полностью обученные цепочки на отложенном test.

## Часть VIII. Финальная проверка на отложенном test

Все предыдущие решения принимались на dev: выбор признаков, схем подготовки, гиперпараметров kNN и силы регуляризации. **Test не участвовал** в этих вычислениях.

`GridSearchCV(refit=True)` после выбора настроек обучил `best_estimator_` на **всём dev**. Теперь обе готовые модели без `island` получат прогнозы для ранее отложенных 86 объектов. Сравним accuracy, macro-F1 и ошибки отдельных видов.

In [ ]:
best_knn = knn_no_island_search.best_estimator_
best_logreg = logreg_no_island_search.best_estimator_

test_pred_knn = best_knn.predict(X_test_no_island)
test_pred_logreg = best_logreg.predict(X_test_no_island)

test_comparison = pd.DataFrame({
    "Модель": ["kNN", "LogisticRegression"],
    "Accuracy test": [accuracy_score(y_test, test_pred_knn),
                      accuracy_score(y_test, test_pred_logreg)],
    "Macro-F1 test": [f1_score(y_test, test_pred_knn, average="macro"),
                      f1_score(y_test, test_pred_logreg, average="macro")],
})
test_comparison.round(3)

### Итоговые матрицы ошибок

Отдельные средние значения могут быть очень близки. Матрица ошибок позволяет понять, какие классы модель путает на test. При 86 тестовых объектах даже одна ошибка способна заметно изменить F1 отдельного редкого класса.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, preds, name in zip(
    axes, [test_pred_knn, test_pred_logreg], ["kNN", "LogisticRegression"]
):
    ConfusionMatrixDisplay.from_predictions(
        y_test, preds, labels=class_order, cmap="Blues", colorbar=False, ax=ax
    )
    ax.set_title(f"Test: {name}, без island")
plt.tight_layout()
plt.show()

In [ ]:
print("Test: kNN")
print(classification_report(y_test, test_pred_knn,
                            labels=class_order, digits=3, zero_division=0))

In [ ]:
print("Test: LogisticRegression")
print(classification_report(y_test, test_pred_logreg,
                            labels=class_order, digits=3, zero_division=0))

### Итоги проверки

Сравним модели по заранее выбранным метрикам и по ошибкам каждого вида. На данном разбиении логистическая регрессия может показать немного более высокое качество, чем kNN. Это **результат конкретной test-выборки**, а не доказательство, что один алгоритм всегда лучше другого.

Мы не используем test для нового подбора параметров: иначе он перестанет быть независимой проверкой. Важнейший результат исследования — понятный и воспроизводимый путь от исходной таблицы до оцениваемой модели, включая заполнение пропусков, кодирование категорий и корректную CV.

## Часть IX. Что хранит обученный Pipeline и как сохранить две модели

В реальной программе новые прогнозы обычно получают без повторного обучения. Для этого сохраняют обученную **цепочку целиком**: правила заполнения пропусков, параметры масштабирования, список категорий и коэффициенты или обучающие наблюдения классификатора.

Сначала заглянем внутрь обеих обученных моделей. Затем сохраним **и логистическую регрессию, и kNN**, повторно загрузим их и проверим, что прогнозы не изменились.

### Заглянем внутрь обученного Pipeline

`best_logreg` — не только два шага `preprocess` и `model`. Внутри `preprocess` находится `ColumnTransformer`, а в нём — числовой и категориальный `Pipeline` с запомненными параметрами.

Посмотрим на знакомые атрибуты `statistics_`, `mean_`, `scale_`, `categories_`, а также на коэффициенты логистической регрессии `coef_` и имена классов `classes_`.

In [ ]:
# Раскрываем вложенную структуру обученной лучшей логистической регрессии.
fitted_preprocessor = best_logreg.named_steps["preprocess"]
fitted_numeric = fitted_preprocessor.named_transformers_["num"]
fitted_categorical = fitted_preprocessor.named_transformers_["cat"]
fitted_classifier = best_logreg.named_steps["model"]

print("Числовые признаки:", numeric_features)
print("Медианы:", fitted_numeric.named_steps["imputer"].statistics_)
print("Средние:", fitted_numeric.named_steps["scaler"].mean_)
print("Масштабы:", fitted_numeric.named_steps["scaler"].scale_)
print("Категории sex:", fitted_categorical.named_steps["onehot"].categories_)
print("Виды:", fitted_classifier.classes_)
print("Форма матрицы коэффициентов:", fitted_classifier.coef_.shape)

Каждая строка `coef_` отвечает своему классу, а столбцы — признакам **после** преобразования. Взглянем на имена преобразованных признаков и соответствующие коэффициенты. Это связывает сохранённый объект с устройством совместной многоклассовой логистической регрессии.

In [ ]:
transformed_names = fitted_preprocessor.get_feature_names_out()
coefficient_table = pd.DataFrame(
    fitted_classifier.coef_,
    index=fitted_classifier.classes_,
    columns=transformed_names,
)
coefficient_table.round(3)

А что хранит **kNN**? Вместо небольшой матрицы коэффициентов у него есть обучающие объекты после всех преобразований. Для каждого нового пингвина он должен искать соседей в этой сохранённой таблице.

In [ ]:
fitted_knn = best_knn.named_steps["model"]
print("Число сохранённых обучающих объектов:", fitted_knn.n_samples_fit_)
print("Число признаков после преобразования:", fitted_knn.n_features_in_)
print("Число соседей k:", fitted_knn.n_neighbors)
print("Способ голосования:", fitted_knn.weights)
print("Метрика расстояния:", fitted_knn.metric)

In [ ]:
from joblib import dump, load

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)

# Сохраняем две обученные цепочки со всеми преобразованиями.
logreg_path = artifact_dir / "penguins_logreg_pipeline.joblib"
knn_path = artifact_dir / "penguins_knn_pipeline.joblib"

dump(best_logreg, logreg_path)
dump(best_knn, knn_path)

In [ ]:
# Файл kNN часто крупнее, потому что вместе с моделью сохраняются train-объекты.
file_sizes = pd.DataFrame({
    "Модель": ["LogisticRegression", "kNN"],
    "Размер, КБ": [logreg_path.stat().st_size / 1024,
                  knn_path.stat().st_size / 1024],
})
file_sizes.round(1)

### Загружаем обе модели

`load` восстанавливает ранее обученный `Pipeline`: его не нужно заново вызывать `fit`. Для модели kNN сохранённый файл содержит также обучающую матрицу после preprocessing, без которой нельзя искать ближайших соседей. Поэтому размер и чувствительность данных имеют значение при хранении таких моделей.

In [ ]:
loaded_logreg = load(logreg_path)
loaded_knn = load(knn_path)

print("Загружена логистическая регрессия:", type(loaded_logreg).__name__)
print("Загружен kNN:", type(loaded_knn).__name__)
print("Шаги обеих цепочек:", list(loaded_logreg.named_steps))

### Прогнозы для новых объектов

Создадим несколько новых строк с **исходными признаками** — размерами тела и полом. Ни масштабирование, ни one-hot вручную не выполняем: это сделает загруженная цепочка. Один `sex` оставим пропущенным, чтобы проверить работу `SimpleImputer`.

In [ ]:
new_penguins = pd.DataFrame({
    "bill_length_mm": [39.4, 49.6, 47.5],
    "bill_depth_mm": [18.5, 15.6, 19.3],
    "flipper_length_mm": [189.0, 218.0, 198.0],
    "body_mass_g": [3800.0, 5200.0, 3950.0],
    "sex": ["male", "female", np.nan],
})
new_penguins

In [ ]:
# Оба готовых Pipeline получают одну и ту же сырую таблицу новых наблюдений.
logreg_predictions = loaded_logreg.predict(new_penguins)
logreg_probabilities = loaded_logreg.predict_proba(new_penguins)
knn_predictions = loaded_knn.predict(new_penguins)

result_new = new_penguins.copy()
result_new["Прогноз LogReg"] = logreg_predictions
result_new["Прогноз kNN"] = knn_predictions

# Вероятности классов показываем для логистической регрессии.
for class_name, column in zip(loaded_logreg.classes_, logreg_probabilities.T):
    result_new[f"P({class_name})"] = column
result_new.round(3)

### Проверяем модели после загрузки

Сравним вероятности классов до сохранения и после загрузки **для обеих моделей**. На тех же входных данных результаты должны совпадать. Так проверим не просто доступность файла, а восстановление всего обученного состояния.

In [ ]:
logreg_before = best_logreg.predict_proba(new_penguins)
logreg_after = loaded_logreg.predict_proba(new_penguins)
knn_before = best_knn.predict_proba(new_penguins)
knn_after = loaded_knn.predict_proba(new_penguins)

print("LogReg: совпадение вероятностей:", np.allclose(logreg_before, logreg_after))
print("kNN: совпадение вероятностей:", np.allclose(knn_before, knn_after))

Файлы формата `joblib` следует загружать только из **доверенных источников**: механизм сериализации Python потенциально выполняет код при загрузке. Для воспроизводимого применения также важны версии Python и библиотек; сохранённый sklearn-артефакт не гарантированно совместим с любой будущей версией.

---

## Итоги

Мы прошли путь от самостоятельной реализации kNN до полноценной работы с реальными табличными данными. Построили baseline, исследовали распределения и пропуски, научились обучать преобразователи и собирать корректный `Pipeline` для смешанных признаков.

С помощью `GridSearchCV` сравнили kNN и многоклассовую логистическую регрессию, исследовали ошибки на отложенных CV-фолдах и повторили полный цикл для постановки без острова. Итоговое сравнение сделали на ранее не использованном test.

Обе обученные цепочки сохранены в файлы. Логистическая регрессия хранит коэффициенты, а kNN — преобразованные обучающие наблюдения, необходимые для поиска соседей. В обоих случаях вместе с классификатором сохранены правила подготовки входных признаков.